In [32]:
"""
Q2.1 从附件2 推导每台过滤器的"当前固定维护规律"
  - T_M_i: 历史中维护平均间隔（天）
  - T_L_i: 历史大维护平均间隔；若 <2 次大维护，标记为 inf
  - 也计算中位数作为鲁棒估计
输出:
  tables/fixed_maintenance_rule.csv
"""
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    starts = []
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd().resolve())

    for start in starts:
        for p in [start, *start.parents]:
            if (p / "Q1输出").exists() and (p / "Q2输出").exists():
                return p
    raise FileNotFoundError("找不到项目根目录：请在 2026_math_modeling_competition 内运行")


In [33]:
ROOT = find_project_root()
mnt = pd.read_csv(ROOT / "Q1输出/data/maintenance_events.csv", parse_dates=["d"])
print(f"Loaded {len(mnt)} maintenance events")


Loaded 127 maintenance events


In [34]:

rows = []
for i in sorted(mnt["i"].unique()):
    sub = mnt[mnt["i"] == i].sort_values("d")
    m_dates = sub[sub["q"] == "m"]["d"].tolist()
    l_dates = sub[sub["q"] == "l"]["d"].tolist()

    # 中维护间隔
    if len(m_dates) >= 2:
        m_int = np.diff([d.toordinal() for d in m_dates])
        T_M_mean = np.mean(m_int)
        T_M_med = np.median(m_int)
    else:
        T_M_mean = T_M_med = np.nan

    # 大维护间隔
    if len(l_dates) >= 2:
        l_int = np.diff([d.toordinal() for d in l_dates])
        T_L_mean = np.mean(l_int)
        T_L_med = np.median(l_int)
    elif len(l_dates) == 1:
        # 只观测到 1 次，无法推间隔
        T_L_mean = T_L_med = np.nan
    else:
        # 0 次：未来也不做大维护
        T_L_mean = T_L_med = np.inf

    rows.append(dict(i=i, n_m=len(m_dates), n_l=len(l_dates),
                     T_M_mean=T_M_mean, T_M_med=T_M_med,
                     T_L_mean=T_L_mean, T_L_med=T_L_med,
                     last_m=m_dates[-1] if m_dates else pd.NaT,
                     last_l=l_dates[-1] if l_dates else pd.NaT))


In [35]:
rule = pd.DataFrame(rows)
print("\nPer-filter empirical maintenance cadence:")
print(rule.to_string(index=False))


Per-filter empirical maintenance cadence:
 i  n_m  n_l  T_M_mean  T_M_med  T_L_mean  T_L_med     last_m     last_l
 1   10    2 73.111111     62.0     347.0    347.0 2026-03-07 2025-11-14
 2    9    3 80.750000     56.5      57.5     57.5 2026-03-12 2026-01-29
 3   11    2 67.600000     55.5     138.0    138.0 2026-04-05 2025-11-17
 4   13    0 54.416667     57.0       inf      inf 2026-03-12        NaT
 5   12    1 62.181818     60.0       NaN      NaN 2026-04-03 2024-09-21
 6   10    2 73.000000     71.0     105.0    105.0 2026-03-25 2025-02-19
 7   11    2 67.600000     62.0     298.0    298.0 2026-03-25 2026-02-08
 8   13    0 55.583333     54.5       inf      inf 2026-03-31        NaT
 9   11    2 63.400000     62.5     321.0    321.0 2026-02-23 2026-04-01
10   10    3 74.444444     74.0     262.5    262.5 2026-03-26 2026-01-01


In [36]:
rule

,i,n_m,n_l,T_M_mean,T_M_med,T_L_mean,T_L_med,last_m,last_l
0,1,10,2,73.111111,62.0,347.0,347.0,2026-03-07,2025-11-14
1,2,9,3,80.750000,56.5,57.5,57.5,2026-03-12,2026-01-29
2,3,11,2,67.600000,55.5,138.0,138.0,2026-04-05,2025-11-17
3,4,13,0,54.416667,57.0,inf,inf,2026-03-12,NaT
4,5,12,1,62.181818,60.0,NaN,NaN,2026-04-03,2024-09-21
5,6,10,2,73.000000,71.0,105.0,105.0,2026-03-25,2025-02-19
6,7,11,2,67.600000,62.0,298.0,298.0,2026-03-25,2026-02-08
7,8,13,0,55.583333,54.5,inf,inf,2026-03-31,NaT
8,9,11,2,63.400000,62.5,321.0,321.0,2026-02-23,2026-04-01
9,10,10,3,74.444444,74.0,262.5,262.5,2026-03-26,2026-01-01


In [37]:
# 对只有 1 次大维护的台（仅 A5），用其余台的中位 T_L 填充
valid_TL = rule.loc[rule["T_L_mean"].notna() & np.isfinite(rule["T_L_mean"]),
                    "T_L_mean"]
T_L_fallback = np.median(valid_TL) if len(valid_TL) > 0 else 365.0
print(f"\nFallback T_L (median of valid): {T_L_fallback:.1f} days")


Fallback T_L (median of valid): 262.5 days


In [38]:

rule["T_M_use"] = rule["T_M_med"]
rule["T_L_use"] = rule["T_L_med"].copy()
mask_nan = rule["T_L_use"].isna()
rule.loc[mask_nan, "T_L_use"] = T_L_fallback
# 对 0 大维护的保持 inf
rule.loc[np.isinf(rule["T_L_mean"]), "T_L_use"] = np.inf


In [39]:

print("\nFinal 固定维护规律 (T_M_use, T_L_use)：")
print(rule[["i", "n_m", "n_l", "T_M_use", "T_L_use"]].to_string(index=False))

rule.to_csv(ROOT / "Q2输出/tables/fixed_maintenance_rule.csv", index=False)
print(f"\nSaved: {ROOT/'Q2输出/tables/fixed_maintenance_rule.csv'}")


Final 固定维护规律 (T_M_use, T_L_use)：
 i  n_m  n_l  T_M_use  T_L_use
 1   10    2     62.0    347.0
 2    9    3     56.5     57.5
 3   11    2     55.5    138.0
 4   13    0     57.0      inf
 5   12    1     60.0    262.5
 6   10    2     71.0    105.0
 7   11    2     62.0    298.0
 8   13    0     54.5      inf
 9   11    2     62.5    321.0
10   10    3     74.0    262.5

Saved: /Users/hhhhollow/Desktop/2026_math_modeling_competition/Q2输出/tables/fixed_maintenance_rule.csv


In [40]:
"""
Q2.2 扩展回归（加入 γ_m·C_m + γ_l·C_l）+ 训练/验证切分
  训练：2024-04-03 至 2025-09-30
  验证：2025-10-01 至 2026-01-19（保留 2026-01-20 起的缺测段）
  比较: base (无 γ) vs extended (含 γ)  →  验证集 RMSE
输出:
  data/Q2_design_with_C.csv
  tables/Q2_regression_compare.csv
"""
import pandas as pd
import numpy as np
from pathlib import Path


In [41]:
def find_project_root():
    starts = []
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd().resolve())

    for start in starts:
        for p in [start, *start.parents]:
            if (p / "Q1输出").exists() and (p / "Q2输出").exists():
                return p
    raise FileNotFoundError("找不到项目根目录：请在 2026_math_modeling_competition 内运行")

ROOT = find_project_root()
df = pd.read_csv(ROOT / "Q1输出/data/daily_with_vars.csv", parse_dates=["d"])
mnt = pd.read_csv(ROOT / "Q1输出/data/maintenance_events.csv", parse_dates=["d"])

In [42]:
# 为每 (i, d) 添加 C_m, C_l (累计维护次数)
def add_cum(df, mnt):
    out = []
    for i, sub in df.groupby("i"):
        sub = sub.sort_values("d").reset_index(drop=True).copy()
        ev = mnt[mnt["i"] == i].sort_values("d")
        m_dates = ev[ev["q"] == "m"]["d"].values
        l_dates = ev[ev["q"] == "l"]["d"].values
        d_arr = sub["d"].values
        sub["C_m"] = np.array([(m_dates <= d).sum() for d in d_arr])
        sub["C_l"] = np.array([(l_dates <= d).sum() for d in d_arr])
        out.append(sub)
    return pd.concat(out, ignore_index=True)


In [43]:
df = add_cum(df, mnt)
print(f"Added C_m (range {df['C_m'].min()}..{df['C_m'].max()}), "
      f"C_l (range {df['C_l'].min()}..{df['C_l'].max()})")

df.to_csv(ROOT / "Q2输出/data/Q2_design_with_C.csv", index=False)


Added C_m (range 0..13), C_l (range 0..3)


In [44]:
df

,i,d,y_raw,n,y,lo_iqr,hi_iqr,is_outlier,u_m,u_l,H_m7,H_l7,A_m,A_l,t,sin1,cos1,season,C_m,C_l
0,1,2024-04-03,91.470808,21,91.470808,54.065501,131.767160,False,0,0,0,0,NaN,NaN,0.0,0.000000,1.000000,春,0,0
1,1,2024-04-04,91.904416,14,91.904416,54.065501,131.767160,False,0,0,0,0,NaN,NaN,1.0,0.017202,0.999852,春,0,0
2,1,2024-04-05,90.867819,16,90.867819,54.065501,131.767160,False,0,0,0,0,NaN,NaN,2.0,0.034398,0.999408,春,0,0
3,1,2024-04-06,90.904580,10,NaN,54.065501,131.767160,False,0,0,0,0,NaN,NaN,3.0,0.051584,0.998669,春,0,0
4,1,2024-04-07,NaN,0,NaN,54.065501,131.767160,False,0,0,0,0,NaN,NaN,4.0,0.068755,0.997634,春,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7385,10,2026-04-07,51.089526,4,NaN,28.578760,164.059361,False,0,0,0,0,12.0,96.0,734.0,0.060172,0.998188,春,10,3
7386,10,2026-04-08,50.755010,4,NaN,28.578760,164.059361,False,0,0,0,0,13.0,97.0,735.0,0.077334,0.997005,春,10,3
7387,10,2026-04-09,50.963301,4,NaN,28.578760,164.059361,False,0,0,0,0,14.0,98.0,736.0,0.094472,0.995527,春,10,3
7388,10,2026-04-10,50.264604,4,NaN,28.578760,164.059361,False,0,0,0,0,15.0,99.0,737.0,0.111583,0.993755,春,10,3


In [45]:
# 只保留 y 非空
df = df.dropna(subset=["y"]).copy()
df["A_m_f"] = df["A_m"].fillna(0)
df["A_l_f"] = df["A_l"].fillna(0)
df["has_m"] = df["A_m"].notna().astype(int)
df["has_l"] = df["A_l"].notna().astype(int)


In [46]:
# 训练/验证切分
train_end = pd.Timestamp("2025-09-30")
val_start = pd.Timestamp("2025-10-01")
val_end   = pd.Timestamp("2026-01-19")

df_tr = df[df["d"] <= train_end].copy()
df_va = df[(df["d"] >= val_start) & (df["d"] <= val_end)].copy()
print(f"Train: {len(df_tr)} rows  ({df_tr['d'].min().date()} .. {df_tr['d'].max().date()})")
print(f"Valid: {len(df_va)} rows  ({df_va['d'].min().date()} .. {df_va['d'].max().date()})")


Train: 4038 rows  (2024-04-03 .. 2025-09-30)
Valid: 1008 rows  (2025-10-01 .. 2026-01-19)


In [47]:
filters = sorted(df["i"].unique())

def design(df, include_gamma=False):
    parts = [np.ones(len(df))]; names = ["const"]
    # 过滤器 FE
    for i in filters[1:]:
        parts.append((df["i"] == i).astype(float).values); names.append(f"I(i={i})")
    # β_i·t
    for i in filters:
        parts.append(((df["i"] == i).astype(float) * df["t"]).values); names.append(f"t_i{i}")
    # 季节
    parts.append(df["sin1"].values); names.append("sin1")
    parts.append(df["cos1"].values); names.append("cos1")
    # 维护短期 & 漂移
    for col in ["H_m7", "H_l7", "A_m_f", "A_l_f", "has_m", "has_l"]:
        parts.append(df[col].values.astype(float)); names.append(col)
    if include_gamma:
        parts.append(df["C_m"].values.astype(float)); names.append("C_m")
        parts.append(df["C_l"].values.astype(float)); names.append("C_l")
    return np.column_stack(parts), names


In [48]:
def ols_fit(X, y):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    n, k = X.shape
    sigma2 = (resid**2).sum() / (n - k)
    try:
        var = sigma2 * np.linalg.inv(X.T @ X)
        se = np.sqrt(np.diag(var))
    except Exception:
        se = np.full(k, np.nan)
    tstat = beta / np.where(se == 0, np.nan, se)
    r2 = 1 - (resid**2).sum() / ((y - y.mean())**2).sum()
    rmse = np.sqrt((resid**2).mean())
    return beta, se, tstat, r2, rmse

In [49]:
def eval_rmse(beta, X, y):
    r = y - X @ beta
    return np.sqrt((r**2).mean())

In [50]:
results = {}
for tag, inc in [("base", False), ("extended", True)]:
    Xtr, names = design(df_tr, include_gamma=inc)
    ytr = df_tr["y"].values
    beta, se, t, r2_tr, rmse_tr = ols_fit(Xtr, ytr)
    # 验证
    Xva, _ = design(df_va, include_gamma=inc)
    yva = df_va["y"].values
    rmse_va = eval_rmse(beta, Xva, yva)
    results[tag] = dict(beta=beta, names=names, r2_tr=r2_tr, rmse_tr=rmse_tr, rmse_va=rmse_va)
    print(f"\n{tag.upper()}:")
    print(f"  train R² = {r2_tr:.4f}, train RMSE = {rmse_tr:.3f}")
    print(f"  valid RMSE = {rmse_va:.3f}")
    if inc:
        j_m = names.index("C_m"); j_l = names.index("C_l")
        print(f"  γ_m = {beta[j_m]:+.3f} (SE={se[j_m]:.3f}, t={t[j_m]:+.2f})")
        print(f"  γ_l = {beta[j_l]:+.3f} (SE={se[j_l]:.3f}, t={t[j_l]:+.2f})")


BASE:
  train R² = 0.8385, train RMSE = 7.413
  valid RMSE = 12.964

EXTENDED:
  train R² = 0.8516, train RMSE = 7.107
  valid RMSE = 14.168
  γ_m = +9.879 (SE=0.620, t=+15.94)
  γ_l = +22.117 (SE=1.390, t=+15.92)


In [51]:
# 汇总输出
rows = []
for tag in ["base", "extended"]:
    r = results[tag]
    rows.append(dict(model=tag,
                     train_R2=r["r2_tr"], train_RMSE=r["rmse_tr"],
                     valid_RMSE=r["rmse_va"]))
cmp = pd.DataFrame(rows).round(4)
print("\n对比：")
print(cmp)
cmp.to_csv(ROOT / "Q2输出/tables/Q2_regression_compare.csv", index=False)


对比：
      model  train_R2  train_RMSE  valid_RMSE
0      base    0.8385      7.4126     12.9635
1  extended    0.8516      7.1071     14.1683


In [52]:

# 保存 extended 的完整系数
ex = results["extended"]
tab = pd.DataFrame({"var": ex["names"], "beta": ex["beta"]}).round(5)
tab.to_csv(ROOT / "Q2输出/tables/Q2_extended_coeffs.csv", index=False)
print(f"\nSaved: Q2_regression_compare.csv,  Q2_extended_coeffs.csv")



Saved: Q2_regression_compare.csv,  Q2_extended_coeffs.csv


In [55]:
"""
Q2.2b 时序交叉验证选 (γ_m, γ_l)
  - 固定 (γ_m, γ_l) 网格值
  - 在训练集上最小化 Σ(y - γ_m*C_m - γ_l*C_l - Xβ)^2 对 β
  - 在验证集评估 RMSE
  - 选验证 RMSE 最小的 (γ_m, γ_l)
"""
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    starts = []
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd().resolve())

    for start in starts:
        for p in [start, *start.parents]:
            if (p / "Q1输出").exists() and (p / "Q2输出").exists():
                return p
    raise FileNotFoundError("找不到项目根目录：请在 2026_math_modeling_competition 内运行")

ROOT = find_project_root()

In [56]:
df = pd.read_csv(ROOT / "Q2输出/data/Q2_design_with_C.csv", parse_dates=["d"])
df = df.dropna(subset=["y"]).copy()
df["A_m_f"] = df["A_m"].fillna(0)
df["A_l_f"] = df["A_l"].fillna(0)
df["has_m"] = df["A_m"].notna().astype(int)
df["has_l"] = df["A_l"].notna().astype(int)

train_end = pd.Timestamp("2025-09-30")
val_start = pd.Timestamp("2025-10-01")
val_end   = pd.Timestamp("2026-01-19")
df_tr = df[df["d"] <= train_end].copy()
df_va = df[(df["d"] >= val_start) & (df["d"] <= val_end)].copy()

filters = sorted(df["i"].unique())

def design_base(df):
    parts = [np.ones(len(df))]; names = ["const"]
    for i in filters[1:]:
        parts.append((df["i"] == i).astype(float).values); names.append(f"I(i={i})")
    for i in filters:
        parts.append(((df["i"] == i).astype(float) * df["t"]).values); names.append(f"t_i{i}")
    parts.append(df["sin1"].values); names.append("sin1")
    parts.append(df["cos1"].values); names.append("cos1")
    for col in ["H_m7", "H_l7", "A_m_f", "A_l_f", "has_m", "has_l"]:
        parts.append(df[col].values.astype(float)); names.append(col)
    return np.column_stack(parts), names

Xtr, names = design_base(df_tr)
Xva, _     = design_base(df_va)
ytr        = df_tr["y"].values
yva        = df_va["y"].values
Cm_tr, Cl_tr = df_tr["C_m"].values, df_tr["C_l"].values
Cm_va, Cl_va = df_va["C_m"].values, df_va["C_l"].values

# 网格：预期 γ<0 (负损伤)，但也允许 0 和小正值做对比
grid_m = np.array([-3, -2, -1, -0.5, -0.25, 0, 0.25])
grid_l = np.array([-10, -5, -2, -1, 0, 1, 2])

rows = []
best = (None, None, np.inf)
for gm in grid_m:
    for gl in grid_l:
        # 目标 y_adj = y - γ·C
        y_adj_tr = ytr - gm * Cm_tr - gl * Cl_tr
        beta, *_ = np.linalg.lstsq(Xtr, y_adj_tr, rcond=None)
        # 验证集预测：ŷ = X·β + γ·C
        yhat_va = Xva @ beta + gm * Cm_va + gl * Cl_va
        rmse_va = np.sqrt(((yva - yhat_va)**2).mean())
        # 训练集也记
        yhat_tr = Xtr @ beta + gm * Cm_tr + gl * Cl_tr
        rmse_tr = np.sqrt(((ytr - yhat_tr)**2).mean())
        rows.append(dict(gamma_m=gm, gamma_l=gl,
                         rmse_train=rmse_tr, rmse_valid=rmse_va))
        if rmse_va < best[2]:
            best = (gm, gl, rmse_va)

tab = pd.DataFrame(rows).round(4)
print("CV grid results:")
print(tab.pivot_table(index="gamma_m", columns="gamma_l", values="rmse_valid").round(3))
print(f"\nBest: γ_m = {best[0]}, γ_l = {best[1]}, valid RMSE = {best[2]:.3f}")

tab.to_csv(ROOT / "Q2输出/tables/gamma_cv_grid.csv", index=False)

# 使用最佳 γ 重新拟合，作为最终模型
gm, gl, _ = best
y_adj_tr = ytr - gm * Cm_tr - gl * Cl_tr
beta_final, *_ = np.linalg.lstsq(Xtr, y_adj_tr, rcond=None)
yhat_tr = Xtr @ beta_final + gm * Cm_tr + gl * Cl_tr
yhat_va = Xva @ beta_final + gm * Cm_va + gl * Cl_va
r2_tr = 1 - ((ytr-yhat_tr)**2).sum() / ((ytr-ytr.mean())**2).sum()
rmse_tr = np.sqrt(((ytr-yhat_tr)**2).mean())
rmse_va = np.sqrt(((yva-yhat_va)**2).mean())
print(f"\nFinal model:  train R²={r2_tr:.4f}, train RMSE={rmse_tr:.3f}, valid RMSE={rmse_va:.3f}")

# 保存
pd.DataFrame({"var": names, "beta": beta_final}).to_csv(
    ROOT / "Q2输出/tables/Q2_final_coeffs.csv", index=False)
pd.DataFrame([dict(gamma_m=gm, gamma_l=gl,
                   train_R2=r2_tr, train_RMSE=rmse_tr, valid_RMSE=rmse_va)]).to_csv(
    ROOT / "Q2输出/tables/Q2_final_summary.csv", index=False)
print(f"\nSaved: Q2_final_coeffs.csv, Q2_final_summary.csv, gamma_cv_grid.csv")

CV grid results:
gamma_l     -10     -5      -2      -1       0       1       2 
gamma_m                                                        
-3.00    13.727  13.221  13.053  13.021  13.000  12.992  12.996
-2.00    13.768  13.232  13.044  13.005  12.978  12.963  12.961
-1.00    13.819  13.252  13.045  13.000  12.966  12.944  12.935
-0.50    13.848  13.266  13.049  13.000  12.964  12.939  12.926
-0.25    13.863  13.274  13.052  13.002  12.963  12.937  12.922
 0.00    13.879  13.282  13.056  13.004  12.964  12.935  12.919
 0.25    13.895  13.291  13.060  13.006  12.964  12.935  12.917

Best: γ_m = 0.25, γ_l = 2, valid RMSE = 12.917

Final model:  train R²=0.8400, train RMSE=7.380, valid RMSE=12.917

Saved: Q2_final_coeffs.csv, Q2_final_summary.csv, gamma_cv_grid.csv
